# Analyse des mesures de la maquette 2 x 2

Ce notebook est une couche de presentation : toutes les fonctions
vivent dans `analysis.py` (testees dans `tests/test_analysis.py`).
Deposer les releves CSV du firmware dans `data/` avec les noms du
protocole (`protocol.md`), puis executer les sections utiles. Chaque
section retombe sur une demonstration synthetique si le fichier
n'existe pas encore.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from analysis import (
    ab_compare,
    crosstalk_db,
    dispersion_pct,
    load_raw_csv,
    load_scan_csv,
    noise_floor_dbfs,
    q_from_ringdown,
    synth_ringdown,
)

DATA = Path("data")
FS_DEFAULT = 3_780_000.0

## M1 et M2 : facteur de qualite

Q par decrement logarithmique de l'enveloppe (reference). Comparer
`m1_q_*.csv` (sans aimant) et `m2_q_ferrite_*.csv` : le critere pivot
est une chute inferieure a 20 % et Q >= 30 avec la ferrite.

In [ ]:
def q_report(pattern):
    rows = []
    for path in sorted(DATA.glob(pattern)):
        samples, fs = load_raw_csv(path)
        f0, q = q_from_ringdown(samples, fs)
        rows.append((path.name, f0 / 1e3, q))
    if not rows:
        samples = synth_ringdown(217e3, 35.0, FS_DEFAULT)
        f0, q = q_from_ringdown(samples, FS_DEFAULT)
        rows = [("demo synthetique 217 kHz / Q35", f0 / 1e3, q)]
    for name, f_khz, q in rows:
        print(f"{name}: f0 = {f_khz:.1f} kHz, Q = {q:.1f}")
    return rows

m1 = q_report("m1_q_*.csv")
m2 = q_report("m2_q_ferrite_*.csv")

## M4 : SNR a l'entrefer nominal (critere >= 20 dB)

In [ ]:
path = DATA / "m4_snr.csv"
if path.exists():
    rows = load_scan_csv(path)
    snr = np.array([r.snr_db for r in rows])
    fa = np.array([r.fa_hz for r in rows])
    print(f"{len(rows)} scans, SNR median {np.median(snr):.1f} dB, "
          f"min {snr.min():.1f} dB")
    plt.figure(figsize=(8, 3))
    plt.hist(snr, bins=30)
    plt.xlabel("SNR (dB)")
    plt.ylabel("n")
    plt.title("M4")
else:
    print("m4_snr.csv absent, voir protocol.md")

## M5 : diaphonie (critere <= -20 dB)

In [ ]:
path = DATA / "m5_diaphonie.csv"
if path.exists():
    rows = load_scan_csv(path)
    by_sq = {}
    for r in rows:
        by_sq.setdefault(r.sq, []).append(r.amp_mv)
    amps = {sq: np.median(v) for sq, v in by_sq.items()}
    active = max(amps, key=amps.get)
    for sq, amp in sorted(amps.items()):
        if sq != active:
            print(f"S{active} -> S{sq}: {crosstalk_db(amps[active], amp):.1f} dB")
else:
    print("m5_diaphonie.csv absent")

## M6 : dispersion des bobines main (critere <= 3 %)

In [ ]:
path = DATA / "m6_dispersion.csv"
if path.exists():
    rows = load_scan_csv(path)
    freqs = [r.fa_hz for r in rows]
    print(f"dispersion +-{dispersion_pct(freqs):.2f} %")
else:
    print("m6_dispersion.csv absent")

## M7 : approche du N42, distance de parking

In [ ]:
import csv

path = DATA / "m7_parking.csv"
if path.exists():
    d, f = [], []
    for rec in csv.reader(open(path)):
        if rec and not rec[0].startswith(("#", "d")):
            d.append(float(rec[0]))
            f.append(float(rec[1]))
    d, f = np.array(d), np.array(f)
    shift = np.abs(f - f[np.argmax(d)])
    plt.figure(figsize=(8, 3))
    plt.plot(d, shift / 1e3, "o-")
    plt.axhline(2.0, color="r", ls="--")
    plt.xlabel("distance (mm)")
    plt.ylabel("decalage (kHz)")
    plt.title("M7")
else:
    print("m7_parking.csv absent")

## M8 : plancher de bruit par configuration (delta <= 6 dB)

In [ ]:
for path in sorted(DATA.glob("m8_bruit_*.csv")):
    samples, fs = load_raw_csv(path)
    print(f"{path.name}: {noise_floor_dbfs(samples):.1f} dBFS")

## M9 : voies A et B sur les memes ringdowns

Si sigma(fb - fa) < 200 Hz avec un taux de validite eleve sur M4, la
voie B (comparateur + capture) gagne et le choix du MCU s'ouvre
(brief 5.2).

In [ ]:
path = DATA / "m4_snr.csv"
if path.exists():
    rep = ab_compare(load_scan_csv(path))
    print(f"n={rep.n}, fb valides={rep.n_b_valid}, "
          f"biais={rep.bias_hz:.0f} Hz, sigma={rep.sigma_hz:.0f} Hz")
else:
    print("m4_snr.csv absent")